In [2]:
import pandas as pd
import numpy as np
# Datei einlesen
file_path = "03_61243-0001_de.csv"
df = pd.read_csv(file_path, sep=";", encoding="latin1", on_bad_lines="skip")
display(df)

,,,Tabelle: 61243-0001
"Strompreise fÃ¼r Haushalte: Deutschland, Halbjahre,",NaN,NaN,NaN
"Jahresverbrauchsklassen, Preisarten",NaN,NaN,NaN
__________,NaN,NaN,NaN
"Â© Statistisches Bundesamt (Destatis), 2026",NaN,NaN,NaN
Stand: 27.04.2026 / 13:24:39,NaN,NaN,NaN


In [3]:
file_path = "03_61243-0001_de.csv"

df_raw = pd.read_csv(
    file_path,
    sep=";",
    encoding="utf-8-sig",
    skiprows=8,
    skipfooter=3,
    engine="python",
    header=None,
    names=[
        "Kategorie",
        "Preis_ohne_Steuern",
        "Kennzeichen_1",
        "Preis_ohne_USt",
        "Kennzeichen_2",
        "Preis_inkl_Steuern",
        "Kennzeichen_3"
    ]
)

display(df_raw.head(20))

,Kategorie,Preis_ohne_Steuern,Kennzeichen_1,Preis_ohne_USt,Kennzeichen_2,Preis_inkl_Steuern,Kennzeichen_3
0,2019,NaN,NaN,NaN,NaN,NaN,NaN
1,1. Halbjahr,NaN,NaN,NaN,NaN,NaN,NaN
2,unter 1 000 KWh,-,NaN,-,NaN,-,NaN
3,1 000 bis unter 2 500 KWh,-,NaN,-,NaN,-,NaN
4,2 500 bis unter 5 000 KWh,-,NaN,-,NaN,-,NaN
5,5 000 bis unter 15 000 KWh,-,NaN,-,NaN,-,NaN
6,15 000 KWh und mehr,-,NaN,-,NaN,-,NaN
7,Insgesamt,-,NaN,-,NaN,-,NaN
8,2. Halbjahr,NaN,NaN,NaN,NaN,NaN,NaN
9,unter 1 000 KWh,"0,2733",e,"0,3840",e,"0,4570",e


In [4]:
df = df_raw.copy()

# Рік
df["Jahr"] = np.where(df["Kategorie"].astype(str).str.match(r"^\d{4}$"), df["Kategorie"], np.nan)
df["Jahr"] = df["Jahr"].ffill()

# Півріччя
df["Halbjahr"] = np.where(df["Kategorie"].astype(str).str.contains("Halbjahr", na=False), df["Kategorie"], np.nan)
df["Halbjahr"] = df["Halbjahr"].ffill()

# Залишаємо тільки рядки з реальними категоріями споживання
df = df[
    ~df["Kategorie"].astype(str).str.match(r"^\d{4}$") &
    ~df["Kategorie"].astype(str).str.contains("Halbjahr", na=False) &
    (df["Kategorie"].astype(str) != "__________")
].copy()

# Чистимо числа
price_cols = ["Preis_ohne_Steuern", "Preis_ohne_USt", "Preis_inkl_Steuern"]

for col in price_cols:
    df[col] = (
        df[col]
        .astype(str)
        .str.replace(",", ".", regex=False)
        .str.replace("\t", ".", regex=False)
        .str.replace("-", "", regex=False)
    )
    df[col] = pd.to_numeric(df[col], errors="coerce")

display(df)

,Kategorie,Preis_ohne_Steuern,Kennzeichen_1,Preis_ohne_USt,Kennzeichen_2,Preis_inkl_Steuern,Kennzeichen_3,Jahr,Halbjahr
2,unter 1 000 KWh,NaN,NaN,NaN,NaN,NaN,NaN,2019,1. Halbjahr
3,1 000 bis unter 2 500 KWh,NaN,NaN,NaN,NaN,NaN,NaN,2019,1. Halbjahr
4,2 500 bis unter 5 000 KWh,NaN,NaN,NaN,NaN,NaN,NaN,2019,1. Halbjahr
5,5 000 bis unter 15 000 KWh,NaN,NaN,NaN,NaN,NaN,NaN,2019,1. Halbjahr
6,15 000 KWh und mehr,NaN,NaN,NaN,NaN,NaN,NaN,2019,1. Halbjahr
...,...,...,...,...,...,...,...,...,...
100,1 000 bis unter 2 500 KWh,0.3039,e,0.3683,e,0.4383,e,2025,2. Halbjahr
101,2 500 bis unter 5 000 KWh,0.2625,e,0.3252,e,0.3869,e,2025,2. Halbjahr
102,5 000 bis unter 15 000 KWh,0.2325,e,0.2922,e,0.3476,e,2025,2. Halbjahr
103,15 000 KWh und mehr,0.2168,e,0.2759,e,0.3283,e,2025,2. Halbjahr


In [5]:
df = df.dropna(subset=["Preis_inkl_Steuern"])

In [6]:
df = df.drop(columns=["Kennzeichen_1", "Kennzeichen_2", "Kennzeichen_3"])

In [7]:
display(df)

,Kategorie,Preis_ohne_Steuern,Preis_ohne_USt,Preis_inkl_Steuern,Jahr,Halbjahr
9,unter 1 000 KWh,0.2733,0.3840,0.4570,2019,2. Halbjahr
10,1 000 bis unter 2 500 KWh,0.1621,0.2726,0.3244,2019,2. Halbjahr
11,2 500 bis unter 5 000 KWh,0.1321,0.2418,0.2878,2019,2. Halbjahr
12,5 000 bis unter 15 000 KWh,0.1139,0.2222,0.2644,2019,2. Halbjahr
13,15 000 KWh und mehr,0.0936,0.1992,0.2370,2019,2. Halbjahr
...,...,...,...,...,...,...
100,1 000 bis unter 2 500 KWh,0.3039,0.3683,0.4383,2025,2. Halbjahr
101,2 500 bis unter 5 000 KWh,0.2625,0.3252,0.3869,2025,2. Halbjahr
102,5 000 bis unter 15 000 KWh,0.2325,0.2922,0.3476,2025,2. Halbjahr
103,15 000 KWh und mehr,0.2168,0.2759,0.3283,2025,2. Halbjahr


In [8]:
def map_halfyear(row):
    if row["Halbjahr"].startswith("1"):
        return f"{row['Jahr']}-01-01"
    else:
        return f"{row['Jahr']}-07-01"

df["date_from"] = pd.to_datetime(df.apply(map_halfyear, axis=1))

In [9]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

In [10]:
df["kategorie"] = df["kategorie"].str.replace(" ", "")

In [11]:
df.columns

Index(['kategorie', 'preis_ohne_steuern', 'preis_ohne_ust',
       'preis_inkl_steuern', 'jahr', 'halbjahr', 'date_from'],
      dtype='object')

In [12]:
df["price"] = df["preis_inkl_steuern"]

In [13]:
df = df.rename(columns={
    "preis_inkl_steuern": "price_eur_per_kwh"
})

In [14]:
def map_halfyear(row):
    if row["halbjahr"].startswith("1"):
        return f"{row['jahr']}-01-01"
    else:
        return f"{row['jahr']}-07-01"

df["date_from"] = pd.to_datetime(df.apply(map_halfyear, axis=1))

In [15]:
df_final = df[[
    "date_from",
    "kategorie",
    "price_eur_per_kwh"
]]

In [16]:
display(df)

,kategorie,preis_ohne_steuern,preis_ohne_ust,price_eur_per_kwh,jahr,halbjahr,date_from,price
9,unter1000KWh,0.2733,0.3840,0.4570,2019,2. Halbjahr,2019-07-01,0.4570
10,1000bisunter2500KWh,0.1621,0.2726,0.3244,2019,2. Halbjahr,2019-07-01,0.3244
11,2500bisunter5000KWh,0.1321,0.2418,0.2878,2019,2. Halbjahr,2019-07-01,0.2878
12,5000bisunter15000KWh,0.1139,0.2222,0.2644,2019,2. Halbjahr,2019-07-01,0.2644
13,15000KWhundmehr,0.0936,0.1992,0.2370,2019,2. Halbjahr,2019-07-01,0.2370
...,...,...,...,...,...,...,...,...
100,1000bisunter2500KWh,0.3039,0.3683,0.4383,2025,2. Halbjahr,2025-07-01,0.4383
101,2500bisunter5000KWh,0.2625,0.3252,0.3869,2025,2. Halbjahr,2025-07-01,0.3869
102,5000bisunter15000KWh,0.2325,0.2922,0.3476,2025,2. Halbjahr,2025-07-01,0.3476
103,15000KWhundmehr,0.2168,0.2759,0.3283,2025,2. Halbjahr,2025-07-01,0.3283


In [17]:
df_final = df[[
    "date_from",
    "jahr",
    "halbjahr",
    "kategorie",
    "preis_ohne_steuern",
    "preis_ohne_ust",
    "price_eur_per_kwh"
]]

In [18]:
display(df_final)

,date_from,jahr,halbjahr,kategorie,preis_ohne_steuern,preis_ohne_ust,price_eur_per_kwh
9,2019-07-01,2019,2. Halbjahr,unter1000KWh,0.2733,0.3840,0.4570
10,2019-07-01,2019,2. Halbjahr,1000bisunter2500KWh,0.1621,0.2726,0.3244
11,2019-07-01,2019,2. Halbjahr,2500bisunter5000KWh,0.1321,0.2418,0.2878
12,2019-07-01,2019,2. Halbjahr,5000bisunter15000KWh,0.1139,0.2222,0.2644
13,2019-07-01,2019,2. Halbjahr,15000KWhundmehr,0.0936,0.1992,0.2370
...,...,...,...,...,...,...,...
100,2025-07-01,2025,2. Halbjahr,1000bisunter2500KWh,0.3039,0.3683,0.4383
101,2025-07-01,2025,2. Halbjahr,2500bisunter5000KWh,0.2625,0.3252,0.3869
102,2025-07-01,2025,2. Halbjahr,5000bisunter15000KWh,0.2325,0.2922,0.3476
103,2025-07-01,2025,2. Halbjahr,15000KWhundmehr,0.2168,0.2759,0.3283


In [19]:
df_final = df_final.rename(columns={
    "date_from": "datum_von",
    "jahr": "jahr",
    "halbjahr": "halbjahr",
    "kategorie": "verbrauchskategorie",
    "preis_ohne_steuern": "strompreis_ohne_steuern_eur_kwh",
    "preis_ohne_ust": "strompreis_ohne_ust_eur_kwh",
    "price_eur_per_kwh": "strompreis_gesamt_eur_kwh"
})

In [20]:
df_final = df_final[
    [
        "datum_von",
        "jahr",
        "halbjahr",
        "verbrauchskategorie",
        "strompreis_ohne_steuern_eur_kwh",
        "strompreis_ohne_ust_eur_kwh",
        "strompreis_gesamt_eur_kwh"
    ]
]

In [21]:
df_final["verbrauchskategorie"] = df_final["verbrauchskategorie"].replace({
    "unter1000KWh": "Unter 1000 kWh",
    "1000bisunter2500KWh": "1000 bis unter 2500 kWh",
    "2500bisunter5000KWh": "2500 bis unter 5000 kWh",
    "5000bisunter15000KWh": "5000 bis unter 15000 kWh",
    "15000KWhundmehr": "15000 kWh und mehr",
    "Insgesamt": "Insgesamt"
})

In [22]:
display(df_final)

,datum_von,jahr,halbjahr,verbrauchskategorie,strompreis_ohne_steuern_eur_kwh,strompreis_ohne_ust_eur_kwh,strompreis_gesamt_eur_kwh
9,2019-07-01,2019,2. Halbjahr,Unter 1000 kWh,0.2733,0.3840,0.4570
10,2019-07-01,2019,2. Halbjahr,1000 bis unter 2500 kWh,0.1621,0.2726,0.3244
11,2019-07-01,2019,2. Halbjahr,2500 bis unter 5000 kWh,0.1321,0.2418,0.2878
12,2019-07-01,2019,2. Halbjahr,5000 bis unter 15000 kWh,0.1139,0.2222,0.2644
13,2019-07-01,2019,2. Halbjahr,15000 kWh und mehr,0.0936,0.1992,0.2370
...,...,...,...,...,...,...,...
100,2025-07-01,2025,2. Halbjahr,1000 bis unter 2500 kWh,0.3039,0.3683,0.4383
101,2025-07-01,2025,2. Halbjahr,2500 bis unter 5000 kWh,0.2625,0.3252,0.3869
102,2025-07-01,2025,2. Halbjahr,5000 bis unter 15000 kWh,0.2325,0.2922,0.3476
103,2025-07-01,2025,2. Halbjahr,15000 kWh und mehr,0.2168,0.2759,0.3283


In [23]:
df_final.to_csv(
    "03_bereinigte_strompreise_LV.csv",
    index=False,
    encoding="utf-8-sig"
)